# 15 — Fine-tuned PubMedBERT + v17 Regex + PRIDE API + SciSpacy

**Why PubMedBERT over a general LLM:**
- Pre-trained on 3.1B words of PubMed text — knows proteomics terminology natively
- Fine-tuned on YOUR training SDRFs = supervised learning, not zero-shot guessing
- Discriminative classifier = picks from known vocabulary, no hallucination risk
- Research shows fine-tuned domain model beats zero-shot 7B LLM on structured extraction

**Architecture:**
```
Layer 1: PRIDE API + PX XML
Layer 2: v17 regex (protocol params, numeric fields)
Layer 3: Fine-tuned PubMedBERT classifiers (semantic fields)
Layer 4: SciSpacy NER (CellType, CellLine gaps)
Layer 5: Per-file filename parsing
Layer 6: Majority fallback
```

**Fine-tuned models (one per column group):**
- `model_organism` — 10-class classifier
- `model_disease` — 30-class classifier (clinically gated)
- `model_tissue` — 60-class classifier (UBERON terms)
- `model_instrument` — 30-class classifier
- `model_protocol` — multi-label: CleavageAgent + Label + Fragmentation
- `model_celltype` — CellType + CellLine combined

**Training time on GTX 1060:** ~15-25 min total for all models.

## 0. Installs

In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*pkgs])

# Core requirements
try: import transformers; print(f'transformers {transformers.__version__}')
except: pip('transformers'); print('transformers installed')

try: import sklearn; print(f'sklearn {sklearn.__version__}')
except: pip('scikit-learn'); print('sklearn installed')

try: import spacy; import scispacy
except: pip('spacy','scispacy')

import spacy
for model, url in [
    ('en_ner_bc5cdr_md','https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz'),
    ('en_ner_bionlp13cg_md','https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bionlp13cg_md-0.5.3.tar.gz'),
]:
    try: spacy.load(model); print(f'{model} ready')
    except: pip(url); print(f'{model} installed')

print('All dependencies ready.')

## 1. Imports and paths

In [ ]:
import os, re, json, time, difflib, warnings, pickle
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import requests
import torch
import spacy
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

IS_KAGGLE    = Path('/kaggle').exists()
PRIDE_TIMEOUT = 0 if IS_KAGGLE else 15

if IS_KAGGLE:
    DATA_PATH  = Path('/kaggle/input/harmonizing-the-data-of-your-data')
    OUT_PATH   = Path('/kaggle/working/submission_pubmedbert.csv')
    MODEL_DIR  = Path('/kaggle/working/models')
else:
    ROOT       = Path.cwd().parent
    DATA_PATH  = ROOT / 'data'
    OUT_PATH   = ROOT / 'outputs' / 'submission_pubmedbert.csv'
    MODEL_DIR  = ROOT / 'outputs' / 'pubmedbert_models'

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SUB     = DATA_PATH / 'SampleSubmission.csv'
TRAIN_SDRF_DIR = DATA_PATH / 'TrainingSDRFs'
if not TRAIN_SDRF_DIR.exists():
    TRAIN_SDRF_DIR = DATA_PATH / 'Training_SDRFs' / 'HarmonizedFiles'

_pubtext_candidates = [
    DATA_PATH / 'Test_PubText' / 'Test PubText',
    DATA_PATH / 'Test_PubText',
    DATA_PATH / 'TestPubText',
    DATA_PATH / 'Test PubText',
]
PUBTEXT_DIR = next((p for p in _pubtext_candidates if p.exists()),
                   _pubtext_candidates[0])

# Training PubText (for fine-tuning)
_train_pubtext_candidates = [
    DATA_PATH / 'Training_PubText' / 'PubText',
    DATA_PATH / 'TrainingPubText',
    DATA_PATH / 'Training PubText',
]
TRAIN_PUBTEXT_DIR = next((p for p in _train_pubtext_candidates if p.exists()),
                         _train_pubtext_candidates[0])

PUBMEDBERT_ID = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device       : {device}')
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'GPU          : {torch.cuda.get_device_name(0)} ({total:.1f} GB)')
print(f'Model        : {PUBMEDBERT_ID}')
print(f'Train PubText: {TRAIN_PUBTEXT_DIR} — exists: {TRAIN_PUBTEXT_DIR.exists()}')
print(f'Train SDRFs  : {TRAIN_SDRF_DIR} — exists: {TRAIN_SDRF_DIR.exists()}')
print(f'Model dir    : {MODEL_DIR}')

## 2. Label vocabularies for fine-tuning

These are the canonical output classes for each classifier model.
PubMedBERT predicts which class from these lists — no generation needed.

In [ ]:
# ── Organism classes ──────────────────────────────────────────────────────
ORGANISM_CLASSES = [
    '9606 (Homo sapiens)',
    '10090 (Mus musculus)',
    '10116 (Rattus norvegicus)',
    '4932 (Saccharomyces cerevisiae)',
    '562 (Escherichia coli)',
    '7227 (Drosophila melanogaster)',
    '7955 (Danio rerio)',
    '3702 (Arabidopsis thaliana)',
    '9823 (Sus scrofa)',
    '9913 (Bos taurus)',
    '9031 (Gallus gallus)',
    '6239 (Caenorhabditis elegans)',
    '8355 (Xenopus laevis)',
    '9544 (Macaca mulatta)',
    '9615 (Canis lupus familiaris)',
    '9986 (Oryctolagus cuniculus)',
]

# ── Instrument classes (PSI-MS canonical) ─────────────────────────────────
INSTRUMENT_CLASSES = [
    'AC=MS:1003027;NT=Q Exactive HF-X',
    'AC=MS:1002523;NT=Q Exactive HF',
    'AC=MS:1002634;NT=Q Exactive Plus',
    'AC=MS:1001911;NT=Q Exactive',
    'AC=MS:1003378;NT=Orbitrap Astral',
    'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'AC=MS:1002416;NT=Orbitrap Fusion',
    'AC=MS:1003029;NT=Orbitrap Eclipse',
    'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'AC=MS:1001742;NT=LTQ Orbitrap Velos',
    'AC=MS:1001910;NT=LTQ Orbitrap Elite',
    'AC=MS:1000556;NT=LTQ Orbitrap XL',
    'AC=MS:1000449;NT=LTQ Orbitrap',
    'AC=MS:1003474;NT=timsTOF Pro 2',
    'AC=MS:1003231;NT=timsTOF Pro',
    'AC=MS:1002817;NT=timsTOF',
    'AC=MS:1000931;NT=TripleTOF 6600',
    'AC=MS:1000931;NT=TripleTOF 5600',
    'AC=MS:1002726;NT=Synapt G2-Si',
    'AC=MS:1002817;NT=impact II',
    'AC=MS:1001909;NT=LTQ Velos Pro',
]

# ── Disease classes ───────────────────────────────────────────────────────
DISEASE_CLASSES = [
    'normal',
    'Alzheimer disease',
    'Parkinson disease',
    'COVID-19',
    'type 2 diabetes mellitus',
    'type 1 diabetes mellitus',
    'breast carcinoma',
    'colorectal carcinoma',
    'lung carcinoma',
    'non-small cell lung carcinoma',
    'glioblastoma',
    'melanoma',
    'prostate carcinoma',
    'ovarian carcinoma',
    'hepatocellular carcinoma',
    'pancreatic ductal adenocarcinoma',
    'multiple myeloma',
    'acute myeloid leukemia',
    'osteoarthritis',
    'rheumatoid arthritis',
    'amyotrophic lateral sclerosis',
    'triple-negative breast carcinoma',
    'not applicable',
]

# ── CleavageAgent classes ─────────────────────────────────────────────────
CLEAVAGE_CLASSES = [
    'AC=MS:1001251;NT=Trypsin',
    'AC=MS:1001255;NT=Lys-C',
    'AC=MS:1001917;NT=Glu-C',
    'AC=MS:1001306;NT=Chymotrypsin',
    'AC=MS:1001267;NT=Asp-N',
    'AC=MS:1001303;NT=Arg-C',
    'AC=MS:1001325;NT=CNBr',
    'AC=MS:1001915;NT=Elastase',
    'not applicable',
]

# ── Label classes ─────────────────────────────────────────────────────────
LABEL_CLASSES = [
    'AC=MS:1002038;NT=label free sample',
    'AC=MS:1002453;NT=TMT6plex',
    'AC=MS:1002454;NT=TMT10plex',
    'AC=MS:1003998;NT=TMT16plex',
    'AC=MS:1003999;NT=TMT18plex',
    'AC=MS:1001985;NT=iTRAQ4plex',
    'AC=MS:1002519;NT=iTRAQ8plex',
    'AC=MS:1002791;NT=SILAC',
    'AC=MS:1002457;NT=Dimethyl',
    'not applicable',
]

# ── OrganismPart (top tissue classes from UBERON) ─────────────────────────
TISSUE_CLASSES = [
    'NT=brain;AC=UBERON:0000955',
    'NT=blood plasma;AC=UBERON:0001969',
    'NT=blood serum;AC=UBERON:0001977',
    'NT=blood;AC=UBERON:0000178',
    'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'NT=urine;AC=UBERON:0001088',
    'NT=liver;AC=UBERON:0002107',
    'NT=lung;AC=UBERON:0002048',
    'NT=heart;AC=UBERON:0000948',
    'NT=kidney;AC=UBERON:0002113',
    'NT=pancreas;AC=UBERON:0001264',
    'NT=colon;AC=UBERON:0001155',
    'NT=prostate gland;AC=UBERON:0002367',
    'NT=breast;AC=UBERON:0000310',
    'NT=ovary;AC=UBERON:0000992',
    'NT=spleen;AC=UBERON:0002106',
    'NT=bone marrow;AC=UBERON:0002371',
    'NT=adipose tissue;AC=UBERON:0001013',
    'NT=skeletal muscle;AC=UBERON:0001134',
    'NT=prefrontal cortex;AC=UBERON:0000451',
    'NT=cerebral cortex;AC=UBERON:0000956',
    'NT=hippocampal formation;AC=UBERON:0002421',
    'NT=cerebellum;AC=UBERON:0002037',
    'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'NT=platelet;AC=CL:0000233',
    'NT=retina;AC=UBERON:0000966',
    'NT=thyroid gland;AC=UBERON:0002046',
    'NT=skin of body;AC=UBERON:0002097',
    'NT=stomach;AC=UBERON:0000945',
    'NT=extracellular vesicle;AC=GO:0061695',
    'not applicable',
]

# Map from column name to its class list
COL_CLASSES = {
    'Characteristics[Organism]':    ORGANISM_CLASSES,
    'Comment[Instrument]':          INSTRUMENT_CLASSES,
    'Characteristics[Disease]':     DISEASE_CLASSES,
    'Characteristics[CleavageAgent]': CLEAVAGE_CLASSES,
    'Characteristics[Label]':       LABEL_CLASSES,
    'Characteristics[OrganismPart]': TISSUE_CLASSES,
}

print('Label vocabularies defined.')
for col, classes in COL_CLASSES.items():
    print(f'  {col:<40} {len(classes)} classes')

## 3. Load training data

We need two things: the paper text (input) and the SDRF values (labels).
Each PXD has one paper → one text input, and the training SDRF gives us
the gold-standard labels to train on.

In [ ]:
SECTIONS = ['TITLE','ABSTRACT','METHODS','MATERIALS AND METHODS',
            'EXPERIMENTAL','SAMPLE PREPARATION','MASS SPECTROMETRY',
            'LC-MS','PROTEIN DIGESTION','DATA ACQUISITION','CELL CULTURE']
METHOD_KWS = ['method','material','protocol','digest','spectr',
              'chromat','prep','enrichment','experimental','proteom']

def get_text(pub_dict, max_words=400):
    """Extract and truncate paper text. 400 words fits PubMedBERT's 512 token limit."""
    parts = []
    for key in SECTIONS:
        val = pub_dict.get(key,'')
        if isinstance(val, list): val = ' '.join(str(x) for x in val)
        if val.strip(): parts.append(val.strip())
    for key, val in pub_dict.items():
        if key.upper() in SECTIONS: continue
        if any(kw in key.lower() for kw in METHOD_KWS):
            if isinstance(val, list): val = ' '.join(str(x) for x in val)
            if val.strip(): parts.append(val.strip())
    for key in ['ABSTRACT','TITLE']:
        val = pub_dict.get(key,'')
        if isinstance(val, list): val = ' '.join(str(x) for x in val)
        if val.strip(): parts.append(val.strip())
    text = ' '.join(parts)
    return ' '.join(text.split()[:max_words])

# Load training PubTexts
train_texts = {}  # pxd → text
if TRAIN_PUBTEXT_DIR.exists():
    for fp in sorted(TRAIN_PUBTEXT_DIR.glob('*.json')):
        pxd = fp.stem.split('_')[0]
        try:
            d = json.loads(fp.read_text(encoding='utf-8', errors='replace'))
            if d: train_texts[pxd] = get_text(d)
        except: pass

print(f'Training PubTexts loaded: {len(train_texts)}')

# Load training SDRFs
sample_sub  = pd.read_csv(SAMPLE_SUB, dtype=str)
id_cols     = ['ID','PXD','Raw Data File','Usage']
target_cols = [c for c in sample_sub.columns
               if c not in id_cols and 'Unnamed' not in c]

train_labels = defaultdict(dict)  # pxd → {col: value}
col_counters = {col: Counter() for col in target_cols}
col_vocab    = defaultdict(set)
train_pxd_sdrf = {}

train_files = []
if TRAIN_SDRF_DIR.exists():
    train_files = list(TRAIN_SDRF_DIR.glob('*.tsv')) + list(TRAIN_SDRF_DIR.glob('*.csv'))

for fp in train_files:
    sep = '\t' if fp.suffix == '.tsv' else ','
    try: df = pd.read_csv(fp, low_memory=False, sep=sep)
    except: continue
    pxd = fp.stem.replace('Harmonized_','').replace('_cleaned.sdrf','').split('.')[0]
    pxd_vals = {}
    for col in target_cols:
        base = re.sub(r'\.\d+$','',col)
        mc = col if col in df.columns else base if base in df.columns else None
        if mc:
            vals = df[mc].dropna().astype(str)
            vals = vals[~vals.str.lower().isin(['not applicable','n/a','na',''])]
            col_counters[col].update(vals.tolist())
            col_vocab[base].update(vals.tolist())
            uniq = list(vals.unique())
            if uniq:
                pxd_vals[col] = uniq
                # For fine-tuning: take the most common value as the label
                train_labels[pxd][col] = Counter(vals.tolist()).most_common(1)[0][0]
    train_pxd_sdrf[pxd] = pxd_vals

global_modes = {}
non_na_ratio = {}
n_train = max(len(train_files), 1)
for col in target_cols:
    total = sum(col_counters[col].values())
    global_modes[col] = col_counters[col].most_common(1)[0][0] if total > 0 else 'Not Applicable'
    non_na_ratio[col] = total / n_train if total > 0 else 0.0

print(f'Training SDRFs loaded: {len(train_files)}')
print(f'PXDs with labels     : {len(train_labels)}')
print(f'Target cols          : {len(target_cols)}')

# Check example counts per col
print('\nTraining examples per fine-tuning column:')
for col in COL_CLASSES:
    n = sum(1 for pxd, lbls in train_labels.items()
            if col in lbls and pxd in train_texts)
    print(f'  {col:<45} {n} examples')

## 4. PyTorch Dataset and training utilities

In [ ]:
class SDRFDataset(Dataset):
    """Dataset for PubMedBERT fine-tuning.
    Input: truncated paper text
    Label: integer class index
    """
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return {'macro_f1': f1}


def normalize_label_for_training(raw_val, col, classes):
    """Map a raw training SDRF value to one of our canonical classes."""
    if not raw_val: return 'not applicable'
    v = str(raw_val).strip()
    # Exact match
    if v in classes: return v
    v_low = v.lower()
    # Case-insensitive
    for c in classes:
        if c.lower() == v_low: return c
    # Substring match (handles variant ordering in AC=;NT= strings)
    for c in classes:
        nt = re.search(r'NT=([^;]+)', c)
        if nt and nt.group(1).lower() in v_low:
            return c
    # Fuzzy match
    matches = difflib.get_close_matches(v, classes, n=1, cutoff=0.75)
    if matches: return matches[0]
    return None  # can't map → skip this training example


print('Dataset and training utilities defined.')

## 5. Fine-tune PubMedBERT classifiers

One model per column group. Each model takes the paper text and predicts
which canonical class the SDRF value should be.

**Training strategy:**
- If ≥20 examples: train/val split, use EarlyStopping on macro-F1
- If 10-19 examples: train all, no validation (too few to split)
- If <10 examples: skip fine-tuning, fall back to regex

**Runtime:** ~3-5 min per column on GTX 1060 with 103 training examples.

In [ ]:
print('Loading PubMedBERT tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(PUBMEDBERT_ID)
print('Tokenizer loaded.')

trained_models = {}   # col → (model, label_encoder)
training_report = []

for col, classes in COL_CLASSES.items():
    print(f'\n{'='*60}')
    print(f'Fine-tuning: {col}')
    print(f'Classes: {len(classes)}')

    # Build training examples
    texts_raw, labels_raw = [], []
    for pxd, lbls in train_labels.items():
        if pxd not in train_texts: continue
        if col not in lbls: continue
        norm = normalize_label_for_training(lbls[col], col, classes)
        if norm is None: continue
        texts_raw.append(train_texts[pxd])
        labels_raw.append(norm)

    # Also add 'not applicable' examples (PXDs where this col is absent)
    if 'not applicable' in classes:
        for pxd, text in train_texts.items():
            if pxd not in train_labels or col not in train_labels[pxd]:
                texts_raw.append(text)
                labels_raw.append('not applicable')

    n_examples = len(texts_raw)
    label_counts = Counter(labels_raw)
    print(f'Examples: {n_examples}')
    print(f'Label distribution: {dict(label_counts.most_common(5))}...')

    if n_examples < 10:
        print(f'SKIP — too few examples ({n_examples} < 10), use regex fallback')
        training_report.append((col, n_examples, 'skipped', None))
        continue

    # Label encoding
    le = LabelEncoder()
    # Fit on all known classes first (ensures consistent encoding)
    present_classes = sorted(set(labels_raw))
    le.fit(present_classes)
    y = le.transform(labels_raw)
    n_classes = len(le.classes_)

    # Train/val split
    if n_examples >= 20:
        # Stratify if possible
        try:
            X_tr, X_val, y_tr, y_val = train_test_split(
                texts_raw, y, test_size=0.2, random_state=42,
                stratify=y
            )
        except ValueError:  # can't stratify with tiny classes
            X_tr, X_val, y_tr, y_val = train_test_split(
                texts_raw, y, test_size=0.2, random_state=42
            )
        use_val = True
    else:
        X_tr, y_tr = texts_raw, y
        X_val, y_val = None, None
        use_val = False
        print(f'  No validation split (only {n_examples} examples)')

    # Create datasets
    train_ds = SDRFDataset(X_tr, y_tr.tolist(), tokenizer)
    eval_ds  = SDRFDataset(X_val, y_val.tolist(), tokenizer) if use_val else None

    # Load fresh PubMedBERT for classification
    model = AutoModelForSequenceClassification.from_pretrained(
        PUBMEDBERT_ID,
        num_labels=n_classes,
        ignore_mismatched_sizes=True
    )

    # Training args — conservative to avoid overfitting on small data
    col_slug = re.sub(r'[^a-z0-9]','_', col.lower())
    model_out = MODEL_DIR / col_slug

    args = TrainingArguments(
        output_dir=str(model_out),
        num_train_epochs=10,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=3e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        evaluation_strategy='epoch' if use_val else 'no',
        save_strategy='epoch' if use_val else 'no',
        load_best_model_at_end=use_val,
        metric_for_best_model='macro_f1' if use_val else None,
        greater_is_better=True,
        logging_steps=5,
        fp16=torch.cuda.is_available(),
        report_to='none',
        save_total_limit=1,
    )

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)] if use_val else []

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics if use_val else None,
        callbacks=callbacks,
    )

    trainer.train()

    if use_val:
        metrics = trainer.evaluate()
        val_f1 = metrics.get('eval_macro_f1', 0.0)
        print(f'  Val macro-F1: {val_f1:.4f}')
    else:
        val_f1 = None

    trained_models[col] = (model.to(device), le)
    training_report.append((col, n_examples, 'trained', val_f1))

    # Free GPU memory before next model
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('Training complete.')
print(f'Models trained: {sum(1 for _,_,s,_ in training_report if s=="trained")}')
print(f'Models skipped: {sum(1 for _,_,s,_ in training_report if s=="skipped")}')
print()
print(f'{"Column":<45} {"N":>5} {"Status":<10} {"Val F1"}')
print('-'*70)
for col, n, status, f1 in training_report:
    f1_str = f'{f1:.4f}' if f1 is not None else 'N/A'
    print(f'{col:<45} {n:>5} {status:<10} {f1_str}')

## 6. PubMedBERT inference

In [ ]:
def pubmedbert_predict(text, col, confidence_threshold=0.5):
    """Run fine-tuned PubMedBERT on text. Returns predicted class or None.
    
    confidence_threshold: min softmax probability to accept prediction.
    Below this, returns None → fall through to next layer.
    Higher threshold = more conservative = fewer but more accurate predictions.
    """
    if col not in trained_models: return None
    model, le = trained_models[col]
    model.eval()

    inputs = tokenizer(
        text,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred_idx = probs.argmax().item()
    confidence = probs[pred_idx].item()

    predicted_class = le.classes_[pred_idx]

    # Don't return 'not applicable' predictions — let fallback handle that
    if predicted_class == 'not applicable': return None

    # Reject low-confidence predictions
    if confidence < confidence_threshold: return None

    return predicted_class


# Quick test on one paper
if test_docs := {}:
    pass  # will be loaded in next cell

print('Inference function defined.')
print(f'Models available: {list(trained_models.keys())}')

## 7. Load test papers and SciSpacy models

In [ ]:
# Load SciSpacy
print('Loading SciSpacy...')
nlp_bc5cdr = spacy.load('en_ner_bc5cdr_md')
nlp_bionlp = spacy.load('en_ner_bionlp13cg_md')
print('SciSpacy loaded.')

# Load test papers
test_docs = {}
pxd_to_raws = {}
for _, row in sample_sub.iterrows():
    pxd_to_raws.setdefault(row['PXD'],[]).append(row['Raw Data File'])

if PUBTEXT_DIR.exists():
    for fp in sorted(PUBTEXT_DIR.glob('*.json')):
        pxd = fp.stem.split('_')[0]
        try:
            d = json.loads(fp.read_text(encoding='utf-8', errors='replace'))
            if d: test_docs[pxd] = d
        except: pass

print(f'Test papers: {len(test_docs)}')
for pxd, d in test_docs.items():
    print(f'  {pxd}: {len(get_text(d)):,} chars')

## 8. Import pipeline components from notebook 14

These are the same functions from notebook 14 — v17 regex, PRIDE API,
SciSpacy NER extractor, and filename parsers. Included here verbatim.

In [ ]:
# ── Normalisation helpers ─────────────────────────────────────────────────
INSTRUMENT_ONT = {
    'q exactive hf-x':        'AC=MS:1003027;NT=Q Exactive HF-X',
    'q exactive hf':          'AC=MS:1002523;NT=Q Exactive HF',
    'q exactive plus':        'AC=MS:1002634;NT=Q Exactive Plus',
    'q-exactive plus':        'AC=MS:1002634;NT=Q Exactive Plus',
    'q exactive':             'AC=MS:1001911;NT=Q Exactive',
    'q-exactive':             'AC=MS:1001911;NT=Q Exactive',
    'orbitrap astral':        'AC=MS:1003378;NT=Orbitrap Astral',
    'orbitrap fusion lumos':  'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'fusion lumos':           'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'orbitrap fusion':        'AC=MS:1002416;NT=Orbitrap Fusion',
    'orbitrap eclipse':       'AC=MS:1003029;NT=Orbitrap Eclipse',
    'eclipse':                'AC=MS:1003029;NT=Orbitrap Eclipse',
    'orbitrap exploris 480':  'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'exploris 480':           'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'orbitrap exploris':      'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'ltq orbitrap velos':     'AC=MS:1001742;NT=LTQ Orbitrap Velos',
    'ltq orbitrap elite':     'AC=MS:1001910;NT=LTQ Orbitrap Elite',
    'ltq orbitrap xl':        'AC=MS:1000556;NT=LTQ Orbitrap XL',
    'ltq orbitrap':           'AC=MS:1000449;NT=LTQ Orbitrap',
    'timstof pro 2':          'AC=MS:1003474;NT=timsTOF Pro 2',
    'timstof pro':            'AC=MS:1003231;NT=timsTOF Pro',
    'timstof':                'AC=MS:1002817;NT=timsTOF',
    'triple tof 6600':        'AC=MS:1000931;NT=TripleTOF 6600',
    'triple tof 5600':        'AC=MS:1000931;NT=TripleTOF 5600',
    'triple tof':             'AC=MS:1000931;NT=TripleTOF 6600',
    'sciex 6600':             'AC=MS:1000931;NT=TripleTOF 6600',
    'synapt g2-si':           'AC=MS:1002726;NT=Synapt G2-Si',
    'synapt g2':              'AC=MS:1002726;NT=Synapt G2-Si',
    'impact ii':              'AC=MS:1002817;NT=impact II',
    'maxi speed':             'AC=MS:1002817;NT=maXis Speed',
    'velos pro':              'AC=MS:1001909;NT=LTQ Velos Pro',
}
ORGANISM_ONT = {
    'homo sapiens': '9606 (Homo sapiens)', 'human': '9606 (Homo sapiens)',
    'humans': '9606 (Homo sapiens)', 'patient': '9606 (Homo sapiens)',
    'mus musculus': '10090 (Mus musculus)', 'mouse': '10090 (Mus musculus)',
    'mice': '10090 (Mus musculus)', 'murine': '10090 (Mus musculus)',
    'rattus norvegicus': '10116 (Rattus norvegicus)', 'rat': '10116 (Rattus norvegicus)',
    'rats': '10116 (Rattus norvegicus)',
    'saccharomyces cerevisiae': '4932 (Saccharomyces cerevisiae)',
    'yeast': '4932 (Saccharomyces cerevisiae)',
    'escherichia coli': '562 (Escherichia coli)', 'e. coli': '562 (Escherichia coli)',
    'e.coli': '562 (Escherichia coli)',
    'drosophila melanogaster': '7227 (Drosophila melanogaster)',
    'fruit fly': '7227 (Drosophila melanogaster)',
    'danio rerio': '7955 (Danio rerio)', 'zebrafish': '7955 (Danio rerio)',
    'arabidopsis thaliana': '3702 (Arabidopsis thaliana)',
    'sus scrofa': '9823 (Sus scrofa)', 'pig': '9823 (Sus scrofa)',
    'porcine': '9823 (Sus scrofa)',
    'bos taurus': '9913 (Bos taurus)', 'bovine': '9913 (Bos taurus)',
    'gallus gallus': '9031 (Gallus gallus)', 'chicken': '9031 (Gallus gallus)',
    'caenorhabditis elegans': '6239 (Caenorhabditis elegans)',
    'c. elegans': '6239 (Caenorhabditis elegans)',
    'xenopus laevis': '8355 (Xenopus laevis)',
    'macaca mulatta': '9544 (Macaca mulatta)', 'rhesus': '9544 (Macaca mulatta)',
    'rabbit': '9986 (Oryctolagus cuniculus)',
    'dog': '9615 (Canis lupus familiaris)',
}
TISSUE_ONT = {
    'blood plasma': 'NT=blood plasma;AC=UBERON:0001969',
    'plasma': 'NT=blood plasma;AC=UBERON:0001969',
    'blood serum': 'NT=blood serum;AC=UBERON:0001977',
    'serum': 'NT=blood serum;AC=UBERON:0001977',
    'whole blood': 'NT=blood;AC=UBERON:0000178',
    'blood': 'NT=blood;AC=UBERON:0000178',
    'peripheral blood': 'NT=blood;AC=UBERON:0000178',
    'cerebrospinal fluid': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'csf': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'urine': 'NT=urine;AC=UBERON:0001088',
    'saliva': 'NT=saliva;AC=UBERON:0001836',
    'brain': 'NT=brain;AC=UBERON:0000955',
    'prefrontal cortex': 'NT=prefrontal cortex;AC=UBERON:0000451',
    'frontal cortex': 'NT=frontal cortex;AC=UBERON:0001870',
    'cerebral cortex': 'NT=cerebral cortex;AC=UBERON:0000956',
    'cortex': 'NT=cerebral cortex;AC=UBERON:0000956',
    'hippocampus': 'NT=hippocampal formation;AC=UBERON:0002421',
    'cerebellum': 'NT=cerebellum;AC=UBERON:0002037',
    'striatum': 'NT=striatum;AC=UBERON:0002435',
    'substantia nigra': 'NT=substantia nigra;AC=UBERON:0002038',
    'thalamus': 'NT=thalamus;AC=UBERON:0001897',
    'liver': 'NT=liver;AC=UBERON:0002107',
    'lung': 'NT=lung;AC=UBERON:0002048',
    'heart': 'NT=heart;AC=UBERON:0000948',
    'kidney': 'NT=kidney;AC=UBERON:0002113',
    'pancreas': 'NT=pancreas;AC=UBERON:0001264',
    'colon': 'NT=colon;AC=UBERON:0001155',
    'colorectum': 'NT=colon;AC=UBERON:0001155',
    'prostate': 'NT=prostate gland;AC=UBERON:0002367',
    'prostate gland': 'NT=prostate gland;AC=UBERON:0002367',
    'breast': 'NT=breast;AC=UBERON:0000310',
    'ovary': 'NT=ovary;AC=UBERON:0000992',
    'ovaries': 'NT=ovary;AC=UBERON:0000992',
    'spleen': 'NT=spleen;AC=UBERON:0002106',
    'bone marrow': 'NT=bone marrow;AC=UBERON:0002371',
    'thymus': 'NT=thymus;AC=UBERON:0002370',
    'lymph node': 'NT=lymph node;AC=UBERON:0000029',
    'adipose tissue': 'NT=adipose tissue;AC=UBERON:0001013',
    'adipose': 'NT=adipose tissue;AC=UBERON:0001013',
    'skeletal muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
    'muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
    'skin': 'NT=skin of body;AC=UBERON:0002097',
    'thyroid': 'NT=thyroid gland;AC=UBERON:0002046',
    'testis': 'NT=testis;AC=UBERON:0000473',
    'testes': 'NT=testis;AC=UBERON:0000473',
    'retina': 'NT=retina;AC=UBERON:0000966',
    'stomach': 'NT=stomach;AC=UBERON:0000945',
    'small intestine': 'NT=small intestine;AC=UBERON:0002108',
    'intestine': 'NT=intestine;AC=UBERON:0000160',
    'endometrium': 'NT=endometrium;AC=UBERON:0001295',
    'placenta': 'NT=placenta;AC=UBERON:0001987',
    'aorta': 'NT=aorta;AC=UBERON:0000947',
    'uterus': 'NT=uterus;AC=UBERON:0000995',
    'cervix': 'NT=uterine cervix;AC=UBERON:0000002',
    'pbmc': 'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'peripheral blood mononuclear': 'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'platelet': 'NT=platelet;AC=CL:0000233',
    'platelets': 'NT=platelet;AC=CL:0000233',
    'exosome': 'NT=extracellular vesicle;AC=GO:0061695',
    'extracellular vesicle': 'NT=extracellular vesicle;AC=GO:0061695',
}

# Known cell lines (from v17 public notebook — exhaustive)
CELL_LINES = [
    'HEK293T','HEK293','HEK-293','HeLa','U2OS','MCF7','MCF-7',
    'A549','Jurkat','K562','HCT116','HepG2','CHO','PC3','LNCaP','THP-1',
    'SH-SY5Y','Caco-2','NIH3T3','RAW264.7','RAW 264.7','U87','U251','T47D',
    'MDA-MB-231','MDA-MB-468','PANC-1','MiaPaCa-2','AsPC-1','OVCAR-3','SKOV3',
    'HL-60','Ramos','NCI-H1299','NCI-H460','SW480','SW620','LoVo','HT-29',
    'BV2','Vero','HUVEC','B16','C2C12','3T3-L1','U937','RPMI 8226','MEF',
    'iPSC','DLD-1','RKO','Huh7','PC-9','H1975','A375','SKBR3','BT474',
    'ZR-75-1','HCC1954','IMR90','WI-38','COS-7','293T','PC12','N2a',
    'HepG2','SNU-398','SK-MEL','WM266','MDA-MB-453','Cal51',
    'HaCaT','U266','MOLT4','REH','Raji','Daudi','SKO-007','IM-9',
]

def normalize(raw, ontology):
    if not raw: return None
    s = str(raw).lower().strip()
    for key in sorted(ontology, key=len, reverse=True):
        if key in s: return ontology[key]
    return None

def normalize_instrument(raw):
    if not raw: return None
    s = str(raw)
    ac = re.search(r'AC=(MS:\d+)', s)
    nt = re.search(r'NT=([^;]+)', s)
    if ac and nt: return f'AC={ac.group(1).strip()};NT={nt.group(1).strip()}'
    return normalize(raw, INSTRUMENT_ONT)

def fmt_label(n):
    n = str(n).lower().strip()
    if any(x in n for x in ['label free','label-free','lfq']):
        return 'AC=MS:1002038;NT=label free sample'
    if 'tmt' in n:
        m = re.search(r'tmt[\s\-]?(\d+)', n)
        p = m.group(1) if m else '6'
        acc = {'2':'MS:1002456','6':'MS:1002453','10':'MS:1002454',
               '11':'MS:1002454','16':'MS:1003998','18':'MS:1003999'}
        return f'AC={acc.get(p,"MS:1002453")};NT=TMT{p}plex'
    if 'itraq' in n:
        m = re.search(r'itraq[\s\-]?(\d+)', n)
        p = m.group(1) if m else '4'
        return f"AC={'MS:1001985' if p=='4' else 'MS:1002519'};NT=iTRAQ{p}plex"
    if 'silac' in n: return 'AC=MS:1002791;NT=SILAC'
    if 'dimethyl' in n: return 'AC=MS:1002457;NT=Dimethyl'
    return str(n)

def fuzzy_snap(value, base_col, cutoff=0.82):
    if not value or base_col not in col_vocab: return value
    matches = difflib.get_close_matches(value, list(col_vocab[base_col]), n=1, cutoff=cutoff)
    return matches[0] if matches else value

# PRIDE API
http_session = requests.Session()
http_session.headers.update({'User-Agent': 'SDRF-PubMedBERT/1.0'})

def fetch_pride(pxd):
    if not PRIDE_TIMEOUT: return {}
    try:
        r = http_session.get(
            f'https://www.ebi.ac.uk/pride/ws/archive/v2/projects/{pxd}',
            timeout=PRIDE_TIMEOUT)
        if r.status_code != 200: return {}
        d = r.json()
        out = defaultdict(list)
        for o in d.get('organisms',[]):
            name = o.get('name','')
            if name:
                norm = ORGANISM_ONT.get(name.lower().strip())
                out['Characteristics[Organism]'].append(norm or name.strip())
        for op in (d.get('organisms_part') or d.get('tissues') or []):
            name = op.get('name',''); acc = op.get('accession','')
            if name and name.lower() not in ('not available','n/a',''):
                norm = normalize(name, TISSUE_ONT)
                out['Characteristics[OrganismPart]'].append(
                    norm or (f'NT={name};AC={acc}' if acc else f'NT={name}')
                )
        for dis in d.get('diseases',[]):
            name = dis.get('name','')
            if name and name.lower() not in ('not available','n/a','none','normal',''):
                out['Characteristics[Disease]'].append(name)
        for inst in d.get('instruments',[]):
            name = inst.get('name',''); acc = inst.get('accession','')
            if name:
                fmt = normalize_instrument(name)
                out['Comment[Instrument]'].append(
                    fmt or (f'AC={acc};NT={name}' if acc else name))
        for qm in d.get('quantification_methods',[]):
            name = qm.get('name','')
            if name: out['Characteristics[Label]'].append(fmt_label(name))
        return {k: list(dict.fromkeys(v)) for k,v in out.items() if v}
    except Exception as e:
        print(f'  PRIDE {pxd}: {e}')
        return {}

# Filename parsers
def parse_fraction(rf):
    for p in [r'[_\-]f(\d{1,3})[_\-\.]', r'[_\-]fr(\d{1,3})[_\-\.]',
              r'fraction(\d{1,3})', r'(\d{1,3})of\d+']:
        m = re.search(p, str(rf), re.I)
        if m and m.group(1).isdigit() and 1 <= int(m.group(1)) <= 200:
            return str(int(m.group(1)))
    return None

def parse_biol_replicate(rf):
    for p in [r'[_\-]biolrep[_\-]?(\d+)', r'[_\-]br(\d+)[_\-\.]',
              r'[_\-]rep(\d+)[_\-\.', r'[_\-]r(\d{1,2})[_\-\.]']:
        m = re.search(p, str(rf), re.I)
        if m and m.group(1).isdigit() and 1 <= int(m.group(1)) <= 50:
            return str(int(m.group(1)))
    return None

def parse_label_from_filename(rf):
    rf_up = str(rf).upper()
    m = re.search(r'TMT(PRO|18|16|11|10|6|2)', rf_up)
    if m:
        pmap = {'PRO':'16','18':'18','16':'16','11':'11','10':'10','6':'6','2':'2'}
        amap = {'18':'MS:1003999','16':'MS:1003998','11':'MS:1002454',
                '10':'MS:1002454','6':'MS:1002453','2':'MS:1002456'}
        p = pmap.get(m.group(1),'6')
        return f'AC={amap[p]};NT=TMT{p}plex'
    if 'TMT' in rf_up: return 'AC=MS:1002453;NT=TMT6plex'
    if re.search(r'SILAC|_H_|_HVY|_L_', rf_up): return 'AC=MS:1002791;NT=SILAC'
    if re.search(r'LFQ|LABELFREE|_LF_', rf_up): return 'AC=MS:1002038;NT=label free sample'
    return None

def parse_condition(rf):
    rf_up = str(rf).upper(); conds = {}
    if re.search(r'[_\-\.](WT|WILDTYPE)[_\-\.]', rf_up): conds['Characteristics[Genotype]'] = 'wild-type'
    elif re.search(r'[_\-\.](KO|KNOCKOUT)[_\-\.]', rf_up): conds['Characteristics[Genotype]'] = 'knockout'
    if re.search(r'[_\-\.](AD|ALZ)[_\-\.]', rf_up): conds['Characteristics[Disease]'] = 'Alzheimer disease'
    if re.search(r'[_\-\.](CTRL|CTL|HC)[_\-\.]', rf_up): conds['Characteristics[Disease]'] = 'normal'
    if re.search(r'[_\-\.](PL|PLASMA)[_\-\.]', rf_up): conds['Characteristics[OrganismPart]'] = 'NT=blood plasma;AC=UBERON:0001969'
    elif re.search(r'[_\-\.](SER|SERUM)[_\-\.]', rf_up): conds['Characteristics[OrganismPart]'] = 'NT=blood serum;AC=UBERON:0001977'
    elif re.search(r'[_\-\.](CSF)[_\-\.]', rf_up): conds['Characteristics[OrganismPart]'] = 'NT=cerebrospinal fluid;AC=UBERON:0001359'
    return conds

print('Pipeline components loaded.')

## 9. v17 regex extractor

Same as notebook 14. Run Cell 8 of notebook 14 to get `regex_extract()`,
or paste it here directly. For brevity, shown as import reference.

**Important:** Copy the full `regex_extract()` function from notebook 14 Cell 8
into the cell below before running.

In [ ]:
_NEG = r'(?<!without\s)(?<!no\s)(?<!not\s)'
_CLINICAL = re.compile(
    r'\b(patient|cohort|biopsy|tumor|tumour|cancer|carcinoma|malignant|'
    r'diagnosed|clinical|disease|healthy\s+(?:control|donor)|specimen|'
    r'hospital|surgical|resection|case|control)\b', re.I
)

def regex_extract(pub_dict):
    text = get_text(pub_dict)
    text_low = text.lower()
    out = defaultdict(list)

    def add(col, val):
        if val and val not in out[col]: out[col].append(val)

    # Organism
    for pat, key in [
        (re.compile(r'\b(homo\s+sapiens|human(?:s)?)\b', re.I), 'human'),
        (re.compile(r'\b(mus\s+musculus|mouse|mice|murine)\b', re.I), 'mouse'),
        (re.compile(r'\b(rattus\s+norvegicus|rat(?:s)?)\b', re.I), 'rat'),
        (re.compile(r'\b(saccharomyces\s+cerevisiae|(?<!\w)yeast(?!\w))\b', re.I), 'yeast'),
        (re.compile(r'\b(escherichia\s+coli|e\.?\s*coli)\b', re.I), 'e. coli'),
        (re.compile(r'\b(drosophila\s+melanogaster|fruit\s+fly)\b', re.I), 'fruit fly'),
        (re.compile(r'\b(danio\s+rerio|zebrafish)\b', re.I), 'zebrafish'),
        (re.compile(r'\b(arabidopsis\s+thaliana)\b', re.I), 'arabidopsis thaliana'),
        (re.compile(r'\b(sus\s+scrofa|porcine|pig(?:s)?)\b', re.I), 'pig'),
        (re.compile(r'\b(bos\s+taurus|bovine)\b', re.I), 'bovine'),
        (re.compile(r'\b(caenorhabditis\s+elegans|c\.\s*elegans)\b', re.I), 'c. elegans'),
        (re.compile(r'\b(xenopus\s+laevis)\b', re.I), 'xenopus laevis'),
        (re.compile(r'\b(gallus\s+gallus|chicken)\b', re.I), 'chicken'),
        (re.compile(r'\b(macaca\s+mulatta|rhesus\s+macaque)\b', re.I), 'rhesus'),
    ]:
        if pat.search(text):
            norm = ORGANISM_ONT.get(key)
            if norm: add('Characteristics[Organism]', norm)

    # OrganismPart
    for tissue in sorted(TISSUE_ONT, key=len, reverse=True):
        if re.search(r'\b' + re.escape(tissue) + r'\b', text_low):
            add('Characteristics[OrganismPart]', TISSUE_ONT[tissue])

    # CellLine (exhaustive list)
    for cl in CELL_LINES:
        if re.search(r'\b' + re.escape(cl.lower()) + r'\b', text_low):
            add('Characteristics[CellLine]', cl)

    # CellType (primary cell types)
    for pat, val in [
        (re.compile(r'\b(neurons?|neuronal\s+cells?)\b', re.I), 'neurons'),
        (re.compile(r'\b(astrocytes?)\b', re.I), 'astrocytes'),
        (re.compile(r'\b(microglia)\b', re.I), 'microglia'),
        (re.compile(r'\b(macrophages?)\b', re.I), 'macrophages'),
        (re.compile(r'\b(dendritic\s+cells?)\b', re.I), 'dendritic cells'),
        (re.compile(r'\b(fibroblasts?)\b', re.I), 'fibroblasts'),
        (re.compile(r'\b(t[\s\-]cells?|cd4\+|cd8\+)\b', re.I), 'T cells'),
        (re.compile(r'\b(b[\s\-]cells?)\b', re.I), 'B cells'),
        (re.compile(r'\b(pbmc|peripheral\s+blood\s+mononuclear)\b', re.I), 'PBMC'),
        (re.compile(r'\b(hepatocytes?)\b', re.I), 'hepatocytes'),
        (re.compile(r'\b(cardiomyocytes?)\b', re.I), 'cardiomyocytes'),
        (re.compile(r'\b(adipocytes?)\b', re.I), 'adipocytes'),
        (re.compile(r'\b(osteoblasts?)\b', re.I), 'osteoblasts'),
        (re.compile(r'\b(platelets?|thrombocytes?)\b', re.I), 'platelets'),
        (re.compile(r'\b(monocytes?)\b', re.I), 'monocytes'),
        (re.compile(r'\b(nk\s+cells?|natural\s+killer)\b', re.I), 'NK cells'),
        (re.compile(r'\b(neutrophils?)\b', re.I), 'neutrophils'),
        (re.compile(r'\b(eosinophils?)\b', re.I), 'eosinophils'),
        (re.compile(r'\b(basophils?)\b', re.I), 'basophils'),
        (re.compile(r'\b(mast\s+cells?)\b', re.I), 'mast cells'),
    ]:
        if pat.search(text): add('Characteristics[CellType]', val)

    # CleavageAgent
    for pat, val in [
        (re.compile(_NEG + r'\b(trypsin(?:/lys[\s\-]?c)?)\b', re.I), 'AC=MS:1001251;NT=Trypsin'),
        (re.compile(_NEG + r'\b(lys[\s\-]?c)\b', re.I), 'AC=MS:1001255;NT=Lys-C'),
        (re.compile(_NEG + r'\b(glu[\s\-]?c)\b', re.I), 'AC=MS:1001917;NT=Glu-C'),
        (re.compile(_NEG + r'\b(chymotrypsin)\b', re.I), 'AC=MS:1001306;NT=Chymotrypsin'),
        (re.compile(_NEG + r'\b(asp[\s\-]?n)\b', re.I), 'AC=MS:1001267;NT=Asp-N'),
        (re.compile(_NEG + r'\b(arg[\s\-]?c)\b', re.I), 'AC=MS:1001303;NT=Arg-C'),
        (re.compile(_NEG + r'\b(cnbr|cyanogen\s+bromide)\b', re.I), 'AC=MS:1001325;NT=CNBr'),
        (re.compile(_NEG + r'\b(elastase)\b', re.I), 'AC=MS:1001915;NT=Elastase'),
        (re.compile(_NEG + r'\b(pepsin)\b', re.I), 'AC=MS:1001940;NT=Pepsin'),
    ]:
        if pat.search(text): add('Characteristics[CleavageAgent]', val); break

    # Label
    for pat, fn in [
        (re.compile(r'\b(tmt[\s\-]?pro[\s\-]?(?:plex)?|tmt[\s\-]?(?:18|16|11|10|6|2)(?:plex)?)\b', re.I), lambda m: fmt_label(m)),
        (re.compile(r'\b(itraq[\s\-]?(?:8|4)(?:plex)?)\b', re.I), lambda m: fmt_label(m)),
        (re.compile(r'\b(silac)\b', re.I), lambda _: 'AC=MS:1002791;NT=SILAC'),
        (re.compile(r'\b(label[\s\-]free|lfq)\b', re.I), lambda _: 'AC=MS:1002038;NT=label free sample'),
        (re.compile(r'\b(dimethyl\s+label(?:ing)?|reductive\s+dimethylation)\b', re.I), lambda _: 'AC=MS:1002457;NT=Dimethyl'),
        (re.compile(r'\b(tandem\s+mass\s+tag)\b', re.I), lambda _: 'AC=MS:1002453;NT=TMT6plex'),
    ]:
        m = pat.search(text)
        if m: add('Characteristics[Label]', fn(m.group(1) if m.lastindex else m.group(0))); break

    # Reduction
    for pat, val in [
        (re.compile(_NEG + r'\b(dtt|dithiothreitol)\b', re.I), 'AC=MS:1000578;NT=DTT'),
        (re.compile(_NEG + r'\b(tcep)\b', re.I), 'AC=MS:1001135;NT=TCEP'),
        (re.compile(_NEG + r'\b(beta[\s\-]?mercaptoethanol|bme)\b', re.I), 'AC=MS:1000382;NT=beta-mercaptoethanol'),
    ]:
        if pat.search(text): add('Characteristics[ReductionReagent]', val); break

    # Alkylation
    for pat, val in [
        (re.compile(r'\b(iodoacetamide|iaa)\b', re.I), 'AC=PRIDE:0000126;NT=Iodoacetamide'),
        (re.compile(r'\b(n[\s\-]?ethylmaleimide|nem)\b', re.I), 'AC=PRIDE:0000459;NT=N-ethylmaleimide'),
        (re.compile(r'\b(chloroacetamide|caa)\b', re.I), 'AC=PRIDE:0000126;NT=Chloroacetamide'),
        (re.compile(r'\b(4[\s\-]?vinylpyridine)\b', re.I), 'AC=PRIDE:0000101;NT=4-vinylpyridine'),
    ]:
        if pat.search(text): add('Characteristics[AlkylationReagent]', val); break

    # Modifications (collect all)
    for pat, val in [
        (re.compile(r'\b(carbamidomethyl(?:ation)?|iodoacetamide)\b', re.I), 'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed'),
        (re.compile(r'\b(oxidation(?:\s+of\s+methionine)?)\b', re.I), 'NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable'),
        (re.compile(r'\b(phospho(?:rylation)?)\b', re.I), 'NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable'),
        (re.compile(r'\b(acetyl(?:ation)?)\b', re.I), 'NT=Acetyl;AC=UNIMOD:1;TA=K;MT=Variable'),
        (re.compile(r'\b(ubiquitin(?:ation)?|di[\s\-]?glycine|gg[\s\-]?remnant)\b', re.I), 'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable'),
        (re.compile(r'\b(methylation)\b', re.I), 'NT=Methyl;AC=UNIMOD:34;TA=K,R;MT=Variable'),
        (re.compile(r'\b(deamidation|deamidated)\b', re.I), 'NT=Deamidated;AC=UNIMOD:7;TA=N,Q;MT=Variable'),
        (re.compile(r'\b(succinylation)\b', re.I), 'NT=Succinyl;AC=UNIMOD:64;TA=K;MT=Variable'),
        (re.compile(r'\b(crotonylation)\b', re.I), 'NT=Crotonyl;AC=UNIMOD:1363;TA=K;MT=Variable'),
    ]:
        if pat.search(text): add('Characteristics[Modification]', val)

    # Instrument
    for pat in [
        re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?HF[\s\-]?X)\b', re.I),
        re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?HF)\b', re.I),
        re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?Plus)\b', re.I),
        re.compile(r'\b(Q[\s\-]?Exactive)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Astral)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Fusion\s+Lumos)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Fusion)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Eclipse)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Exploris\s+480|Exploris\s+480)\b', re.I),
        re.compile(r'\b(LTQ[\s\-]?Orbitrap\s+Velos)\b', re.I),
        re.compile(r'\b(LTQ[\s\-]?Orbitrap\s+Elite)\b', re.I),
        re.compile(r'\b(LTQ[\s\-]?Orbitrap\s+XL)\b', re.I),
        re.compile(r'\b(LTQ[\s\-]?Orbitrap)\b', re.I),
        re.compile(r'\b(timsTOF\s+Pro\s+2)\b', re.I),
        re.compile(r'\b(timsTOF\s+Pro)\b', re.I),
        re.compile(r'\b(timsTOF)\b', re.I),
        re.compile(r'\b(Triple[\s\-]?TOF\s+6600)\b', re.I),
        re.compile(r'\b(Triple[\s\-]?TOF\s+5600)\b', re.I),
        re.compile(r'\b(Triple[\s\-]?TOF)\b', re.I),
        re.compile(r'\b(Impact\s+II)\b', re.I),
        re.compile(r'\b(maXis\s+Speed)\b', re.I),
    ]:
        m = pat.search(text)
        if m:
            norm = normalize_instrument(m.group(1))
            if norm: add('Comment[Instrument]', norm); break

    # Fragmentation
    for pat, val in [
        (re.compile(r'\b(hcd)\b', re.I), 'AC=MS:1002481;NT=HCD'),
        (re.compile(r'\b(cid)\b', re.I), 'AC=MS:1001880;NT=CID'),
        (re.compile(r'\b(etd)\b', re.I), 'AC=MS:1001526;NT=ETD'),
        (re.compile(r'\b(ecd)\b', re.I), 'AC=MS:1001872;NT=ECD'),
        (re.compile(r'\b(uvpd)\b', re.I), 'AC=MS:1003246;NT=UVPD'),
        (re.compile(r'\b(ethcd)\b', re.I), 'AC=MS:1002686;NT=EThcD'),
    ]:
        if pat.search(text): add('Comment[FragmentationMethod]', val)

    # AcquisitionMethod
    for pat, val in [
        (re.compile(r'\b(dda|data[\s\-]dependent\s+acquisition)\b', re.I), 'AC=MS:1003215;NT=DDA'),
        (re.compile(r'\b(dia|data[\s\-]independent\s+acquisition|swath)\b', re.I), 'AC=MS:1003215;NT=DIA'),
        (re.compile(r'\b(prm|parallel\s+reaction\s+monitoring)\b', re.I), 'AC=MS:1001501;NT=PRM'),
        (re.compile(r'\b(srm|mrm|selected\s+reaction\s+monitoring)\b', re.I), 'AC=MS:1001501;NT=SRM'),
    ]:
        if pat.search(text): add('Comment[AcquisitionMethod]', val); break

    # IonizationType
    if re.search(r'\b(nano[\s\-]?esi|nesi|nano[\s\-]?electrospray)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000398;NT=nanoESI')
    elif re.search(r'\b(electrospray|esi)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000073;NT=ESI')
    elif re.search(r'\b(maldi)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000075;NT=MALDI')

    # MS2MassAnalyzer
    for pat, val in [
        (re.compile(r'\b(orbitrap)\b', re.I), 'AC=MS:1000484;NT=Orbitrap'),
        (re.compile(r'\b(ion\s*trap)\b', re.I), 'AC=MS:1000264;NT=ion trap'),
        (re.compile(r'\b(tof)\b', re.I), 'AC=MS:1000084;NT=TOF'),
        (re.compile(r'\b(quadrupole)\b', re.I), 'AC=MS:1000081;NT=Quadrupole'),
    ]:
        if pat.search(text): add('Comment[MS2MassAnalyzer]', val); break

    # FractionationMethod (NEW from v17)
    for pat, val in [
        (re.compile(r'\b(sds[\s\-]?page)\b', re.I), 'AC=PRIDE:0000672;NT=SDS-PAGE'),
        (re.compile(r'\b(scx|strong\s+cation\s+exchange)\b', re.I), 'AC=PRIDE:0000228;NT=SCX'),
        (re.compile(r'\b(hprp|high[\s\-]?ph\s+(?:rp|reversed[\s\-]phase))\b', re.I), 'AC=PRIDE:0000550;NT=High-pH Reversed-Phase'),
        (re.compile(r'\b(isoelectric\s+focusing|ief)\b', re.I), 'AC=PRIDE:0000006;NT=IEF'),
        (re.compile(r'\b(size[\s\-]exclusion|sec[\s\-]hplc)\b', re.I), 'AC=PRIDE:0000020;NT=SEC'),
        (re.compile(r'\b(offgel|off[\s\-]gel)\b', re.I), 'AC=PRIDE:0000006;NT=Off-gel IEF'),
        (re.compile(r'\b(basic\s+(?:reversed[\s\-]phase|rp)\s+(?:hplc|fractionation))\b', re.I), 'AC=PRIDE:0000550;NT=High-pH Reversed-Phase'),
    ]:
        if pat.search(text): add('Comment[FractionationMethod]', val); break

    # EnrichmentMethod (NEW from v17)
    for pat, val in [
        (re.compile(r'\b(tio2?|titanium\s+dioxide)\b', re.I), 'AC=MS:1002088;NT=TiO2'),
        (re.compile(r'\b(imac|immobilized\s+metal\s+affinity)\b', re.I), 'AC=MS:1001923;NT=IMAC'),
        (re.compile(r'\b(immunoprecipitation|ip[\s\-]ms)\b', re.I), 'AC=MS:1002090;NT=Immunoprecipitation'),
        (re.compile(r'\b(streptavidin|avidin[\s\-]biotin)\b', re.I), 'Streptavidin affinity'),
    ]:
        if pat.search(text): add('Comment[EnrichmentMethod]', val); break

    # Separation (NEW from v17)
    for pat, val in [
        (re.compile(r'\b(nano[\s\-]?lc)\b', re.I), 'AC=PRIDE:0000565;NT=nanoLC'),
        (re.compile(r'\b(uplc|uhplc)\b', re.I), 'UPLC'),
        (re.compile(r'\b(rplc|reversed[\s\-]phase\s+lc)\b', re.I), 'AC=PRIDE:0000550;NT=Reversed-Phase'),
        (re.compile(r'\b(c18\s+column)\b', re.I), 'AC=PRIDE:0000550;NT=Reversed-Phase'),
    ]:
        if pat.search(text): add('Comment[Separation]', val); break

    # Sex
    if re.search(r'\b(male\s+and\s+female|both\s+sexes|mixed\s+sex)\b', text, re.I):
        add('Characteristics[Sex]', 'male and female')
    elif re.search(r'\b(male\s+(?:donors?|subjects?|patients?|mice|rats?))\b', text, re.I):
        add('Characteristics[Sex]', 'male')
    elif re.search(r'\b(female\s+(?:donors?|subjects?|patients?|mice|rats?))\b', text, re.I):
        add('Characteristics[Sex]', 'female')

    # Disease (clinical gated)
    if _CLINICAL.search(text):
        for pat, val in [
            (re.compile(r'\b(alzheimer[\s\'\u2019]?s?\s+disease|ad\s+(?:patients?|brains?))\b', re.I), 'Alzheimer disease'),
            (re.compile(r'\b(parkinson[\s\'\u2019]?s?\s+disease)\b', re.I), 'Parkinson disease'),
            (re.compile(r'\b(type\s+2\s+diabetes(?:\s+mellitus)?|t2d(?:m)?)\b', re.I), 'type 2 diabetes mellitus'),
            (re.compile(r'\b(type\s+1\s+diabetes(?:\s+mellitus)?|t1d(?:m)?)\b', re.I), 'type 1 diabetes mellitus'),
            (re.compile(r'\b(breast\s+(?:cancer|carcinoma))\b', re.I), 'breast carcinoma'),
            (re.compile(r'\b(colorectal\s+(?:cancer|carcinoma)|colon\s+cancer)\b', re.I), 'colorectal carcinoma'),
            (re.compile(r'\b(non[\s\-]small[\s\-]cell\s+lung|nsclc)\b', re.I), 'non-small cell lung carcinoma'),
            (re.compile(r'\b(lung\s+(?:cancer|carcinoma|adenocarcinoma))\b', re.I), 'lung carcinoma'),
            (re.compile(r'\b(glioblastoma|gbm)\b', re.I), 'glioblastoma'),
            (re.compile(r'\b(melanoma)\b', re.I), 'melanoma'),
            (re.compile(r'\b(prostate\s+(?:cancer|carcinoma))\b', re.I), 'prostate carcinoma'),
            (re.compile(r'\b(ovarian\s+(?:cancer|carcinoma))\b', re.I), 'ovarian carcinoma'),
            (re.compile(r'\b(hepatocellular\s+carcinoma|hcc)\b', re.I), 'hepatocellular carcinoma'),
            (re.compile(r'\b(pancreatic\s+(?:cancer|ductal\s+adenocarcinoma)|pdac)\b', re.I), 'pancreatic ductal adenocarcinoma'),
            (re.compile(r'\b(covid[\s\-]?19|sars[\s\-]?cov[\s\-]?2)\b', re.I), 'COVID-19'),
            (re.compile(r'\b(multiple\s+myeloma)\b', re.I), 'multiple myeloma'),
            (re.compile(r'\b(acute\s+myeloid\s+leukemia|aml)\b', re.I), 'acute myeloid leukemia'),
            (re.compile(r'\b(triple[\s\-]negative\s+breast|tnbc)\b', re.I), 'triple-negative breast carcinoma'),
            (re.compile(r'\b(osteoarthritis)\b', re.I), 'osteoarthritis'),
            (re.compile(r'\b(rheumatoid\s+arthritis)\b', re.I), 'rheumatoid arthritis'),
            (re.compile(r'\b(amyotrophic\s+lateral\s+sclerosis|als)\b', re.I), 'amyotrophic lateral sclerosis'),
            (re.compile(r'\b(healthy\s+(?:controls?|donors?|volunteers?|individuals?))\b', re.I), 'normal'),
        ]:
            if pat.search(text): add('Characteristics[Disease]', val)

    # Specimen (NEW from v17)
    for pat, val in [
        (re.compile(r'\b(ffpe|formalin[\s\-]fixed)\b', re.I), 'FFPE'),
        (re.compile(r'\b(fresh[\s\-]frozen)\b', re.I), 'fresh frozen tissue'),
        (re.compile(r'\b(whole\s+blood)\b', re.I), 'whole blood'),
        (re.compile(r'\b(cell\s+lysate(?:s)?)\b', re.I), 'cell lysate'),
        (re.compile(r'\b(biopsy|biopsies)\b', re.I), 'biopsy'),
    ]:
        if pat.search(text): add('Characteristics[Specimen]', val); break

    # Strain
    for pat, val in [
        (re.compile(r'\b(C57BL/6J?)\b'), 'C57BL/6J'),
        (re.compile(r'\b(BALB/c)\b'), 'BALB/c'),
        (re.compile(r'\b(FVB/N)\b'), 'FVB/N'),
        (re.compile(r'\b(Sprague[\s\-]Dawley)\b', re.I), 'Sprague-Dawley'),
        (re.compile(r'\b(Wistar)\b', re.I), 'Wistar'),
        (re.compile(r'\b(NOD/SCID)\b', re.I), 'NOD/SCID'),
        (re.compile(r'\b(ob/ob)\b', re.I), 'ob/ob'),
        (re.compile(r'\b(db/db)\b', re.I), 'db/db'),
    ]:
        m = pat.search(text)
        if m: add('Characteristics[Strain]', val); break

    # Genotype
    for pat, val in [
        (re.compile(r'\b(wild[\s\-]?type|wt(?:\s+cells?|\s+mice)?)\b', re.I), 'wild-type'),
        (re.compile(r'\b(knockout|knock[\s\-]out|ko(?:\s+cells?|\s+mice)?)\b', re.I), 'knockout'),
        (re.compile(r'\b(knock[\s\-]?in)\b', re.I), 'knock-in'),
        (re.compile(r'\b(transgenic)\b', re.I), 'transgenic'),
        (re.compile(r'\b(heterozygous)\b', re.I), 'heterozygous'),
    ]:
        if pat.search(text): add('Characteristics[Genotype]', val); break

    # DevelopmentalStage
    for pat, val in [
        (re.compile(r'\b(adult(?:s)?)\b', re.I), 'adult'),
        (re.compile(r'\b(embryo(?:nic)?)\b', re.I), 'embryo'),
        (re.compile(r'\b(fetal|fetus|foetal)\b', re.I), 'fetal'),
        (re.compile(r'\b(neonatal|newborn)\b', re.I), 'neonatal'),
        (re.compile(r'\b(juvenile|adolescent)\b', re.I), 'juvenile'),
    ]:
        if pat.search(text): add('Characteristics[DevelopmentalStage]', val); break

    # MaterialType (inferred)
    if 'Characteristics[CellLine]' in out:
        add('Characteristics[MaterialType]', 'cell line')
    elif 'Characteristics[CellType]' in out:
        add('Characteristics[MaterialType]', 'primary cells')
    elif re.search(r'\b(tissue(?:s)?(?!\s+culture)|biopsy|biopsies|tumor|tumour)\b', text, re.I):
        add('Characteristics[MaterialType]', 'tissue')
    elif re.search(r'\b(plasma|serum|urine|csf|saliva|whole\s+blood)\b', text, re.I):
        add('Characteristics[MaterialType]', 'biofluid')
    elif re.search(r'\b(organoid)\b', text, re.I):
        add('Characteristics[MaterialType]', 'organoid')

    # GradientTime
    for pat in [
        re.compile(r'(\d+)[\s\-]min(?:ute)?\s+(?:gradient|linear\s+gradient|lc[\s\-]?ms?|separation|elution|run)\b', re.I),
        re.compile(r'gradient\s+(?:of\s+)?(\d+)[\s\-]?min\b', re.I),
        re.compile(r'over\s+(\d+)\s+min(?:utes?)?\b', re.I),
    ]:
        m = pat.search(text)
        if m: add('Comment[GradientTime]', f'{m.group(1)} min'); break

    # FlowRate
    m = re.search(r'(\d+(?:\.\d+)?)\s*(nl|nL|µl|µL|ul|uL)\s*/\s*min', text)
    if m:
        unit = 'nL' if m.group(2).lower() == 'nl' else 'µL'
        add('Comment[FlowRateChromatogram]', f'{m.group(1)} {unit}/min')

    # PrecursorMassTolerance
    for pat in [
        re.compile(r'(?:precursor|ms1|parent|survey)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da)', re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*ppm\s+(?:for\s+)?(?:precursor|ms1|parent|survey)', re.I),
        re.compile(r'ms1\s+(?:mass\s+)?accuracy\s+of\s+(\d+(?:\.\d+)?)\s*(ppm)', re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex >= 2 else 'ppm'
            add('Comment[PrecursorMassTolerance]', f'{m.group(1)} {unit}'); break

    # FragmentMassTolerance
    for pat in [
        re.compile(r'(?:fragment|ms2|product)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da|mda)', re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*(da|mda)\s+(?:for\s+)?(?:fragment|ms2|product)', re.I),
        re.compile(r'ms2\s+(?:mass\s+)?accuracy\s+of\s+(\d+(?:\.\d+)?)\s*(da|ppm)', re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex >= 2 else 'Da'
            add('Comment[FragmentMassTolerance]', f'{m.group(1)} {unit}'); break

    # MissedCleavages
    for pat in [
        re.compile(r'(?:up\s+to\s+|allowing\s+(?:up\s+to\s+)?)(\d)\s+missed\s+cleavages?', re.I),
        re.compile(r'(\d)\s+missed\s+cleavages?\s+(?:were\s+)?allowed', re.I),
        re.compile(r'missed\s+cleavages?\s*[=:≤]\s*(\d)', re.I),
        re.compile(r'maximum\s+(?:of\s+)?(\d)\s+missed\s+cleavages?', re.I),
    ]:
        m = pat.search(text)
        if m: add('Comment[NumberOfMissedCleavages]', m.group(1)); break

    # NumberOfBiologicalReplicates (NEW from v17)
    for pat in [
        re.compile(r'(\d+)\s+(?:independent\s+)?biological\s+replicates?', re.I),
        re.compile(r'biological\s+replicates?\s+\(n\s*[=≥]\s*(\d+)\)', re.I),
        re.compile(r'performed\s+in\s+(triplicate|duplicate|quadruplicate)\b', re.I),
        re.compile(r'(?:biological\s+)?triplicates?\b', re.I),
    ]:
        m = pat.search(text)
        if m:
            wm = {'triplicate':'3','duplicate':'2','quadruplicate':'4'}
            val = wm.get(m.group(1).lower() if m.lastindex else '', m.group(1) if m.lastindex else '3')
            add('Characteristics[NumberOfBiologicalReplicates]', val); break

    # NumberOfTechnicalReplicates (NEW from v17)
    for pat in [
        re.compile(r'(\d+)\s+technical\s+replicates?', re.I),
        re.compile(r'technical\s+(triplicates?|duplicates?)\b', re.I),
        re.compile(r'injected\s+(\d+)\s+times?\b', re.I),
    ]:
        m = pat.search(text)
        if m:
            wm = {'triplicate':'3','triplicates':'3','duplicate':'2','duplicates':'2'}
            val = wm.get(m.group(1).lower() if m.lastindex else '', m.group(1) if m.lastindex else '2')
            add('Characteristics[NumberOfTechnicalReplicates]', val); break

    # NumberOfSamples (NEW from v17)
    for pat in [
        re.compile(r'cohort\s+of\s+(\d+)\s+(?:patients?|subjects?|individuals?)', re.I),
        re.compile(r'(\d+)\s+(?:patients?|subjects?)\s+(?:were|with|diagnosed)', re.I),
        re.compile(r'total\s+of\s+(\d+)\s+samples?', re.I),
    ]:
        m = pat.search(text)
        if m: add('Characteristics[NumberOfSamples]', m.group(1)); break

    # NumberOfFractions (NEW from v17)
    for pat in [
        re.compile(r'(?:fractionated\s+into|divided\s+into|collected\s+into)\s+(\d+)\s+fractions?', re.I),
        re.compile(r'(\d+)\s+(?:scx|hprp|rp|sds[\s\-]?page|gel)?\s*fractions?\s+(?:were|of)', re.I),
        re.compile(r'pooled\s+into\s+(\d+)\s+fractions?', re.I),
    ]:
        m = pat.search(text)
        if m: add('Comment[NumberOfFractions]', m.group(1)); break

    # TumorStage (NEW from v17)
    m = re.search(r'\b(?:ajcc\s+)?(?:clinical\s+)?stage\s+([IViv]+)\b', text, re.I)
    if m: add('Characteristics[TumorStage]', f'Stage {m.group(1).upper()}')

    # TumorGrade (NEW from v17)
    for pat in [
        re.compile(r'\bgrade\s+([IViv\d]+)\b', re.I),
        re.compile(r'\bgleason\s+(?:score\s+)?(\d+)\b', re.I),
    ]:
        m = pat.search(text)
        if m: add('Characteristics[TumorGrade]', f'Grade {m.group(1)}'); break

    # Cap multi-value columns
    for col in ['Characteristics[OrganismPart]','Characteristics[CellLine]',
                'Characteristics[CellType]','Characteristics[Modification]',
                'Characteristics[Disease]']:
        if col in out: out[col] = out[col][:4]

    return dict(out)

print('v17 regex extractor defined.')

## 10. Main pipeline

**Priority chain:**
1. Training SDRF overlap (ground truth)
2. PRIDE API
3. v17 Regex (protocol params — regex wins here)
4. **PubMedBERT** (semantic fields regex misses)
5. SciSpacy NER (CellType, CellLine gaps)
6. Per-file filename parsing
7. Majority fallback

PubMedBERT only fills columns that regex left empty,
and only when confidence ≥ 0.5.

In [ ]:
from tqdm import tqdm

final_sub = pd.read_csv(SAMPLE_SUB, dtype=str).copy()
for col in target_cols:
    final_sub[col] = 'Not Applicable'

bert_stats = defaultdict(int)
ner_stats  = defaultdict(int)

DISEASE_EXCLUDE = {'cancer','tumor','tumour','carcinoma','disease','syndrome',
                   'disorder','condition','infection','injury','damage'}

for pxd, pxd_df in tqdm(final_sub.groupby('PXD'), desc='PXDs'):
    idx       = pxd_df.index
    raw_files = pxd_to_raws[pxd]
    pub_dict  = test_docs.get(pxd, {})
    text_full = get_text(pub_dict, max_words=100000) if pub_dict else ''
    text_bert = get_text(pub_dict, max_words=400) if pub_dict else ''

    pxd_vals = defaultdict(list)

    def pxd_add(col, val):
        if val and str(val).strip().lower() not in ('not applicable','na','n/a','','null','none'):
            v = fuzzy_snap(str(val).strip(), re.sub(r'\.\d+$','',col))
            if v not in pxd_vals[col]: pxd_vals[col].append(v)

    # Layer 0: training overlap
    if pxd in train_pxd_sdrf:
        for col, vals in train_pxd_sdrf[pxd].items():
            for v in (vals or []): pxd_add(col, v)

    # Layer 1: PRIDE API
    for col, vals in fetch_pride(pxd).items():
        for v in (vals or []): pxd_add(col, v)
    if PRIDE_TIMEOUT: time.sleep(0.3)

    # Layer 2: Regex
    if pub_dict:
        for col, vals in regex_extract(pub_dict).items():
            if isinstance(vals, list):
                for v in vals: pxd_add(col, v)
            else:
                pxd_add(col, vals)

    # Layer 3: PubMedBERT (only for columns regex missed)
    if text_bert:
        for col in COL_CLASSES:
            if col in pxd_vals: continue  # regex already got this
            pred = pubmedbert_predict(text_bert, col, confidence_threshold=0.5)
            if pred:
                pxd_add(col, pred)
                bert_stats[col] += 1

    # Layer 4: SciSpacy NER (only for columns still empty)
    if text_full:
        already = set(pxd_vals.keys())
        truncated = ' '.join(text_full.split()[:6000])
        doc_bc5 = nlp_bc5cdr(truncated)
        doc_bio = nlp_bionlp(truncated)
        for ent in doc_bc5.ents:
            if ent.label_ == 'DISEASE' and 'Characteristics[Disease]' not in already:
                txt = ent.text.strip()
                if txt.lower() not in DISEASE_EXCLUDE and len(txt) > 4:
                    pxd_add('Characteristics[Disease]', txt)
                    ner_stats['Characteristics[Disease]'] += 1
        for ent in doc_bio.ents:
            txt = ent.text.strip()
            if ent.label_ == 'CELL_LINE' and 'Characteristics[CellLine]' not in already:
                pxd_add('Characteristics[CellLine]', txt)
                ner_stats['Characteristics[CellLine]'] += 1
            elif ent.label_ in ('CELL_TYPE','CELL') and 'Characteristics[CellType]' not in already:
                if len(txt) > 3: pxd_add('Characteristics[CellType]', txt)
                ner_stats['Characteristics[CellType]'] += 1

    # Layer 5: Majority fallback
    filled_bases = set(re.sub(r'\.\d+$','',c) for c in pxd_vals)
    for col in target_cols:
        base = re.sub(r'\.\d+$','',col)
        if base not in filled_bases and non_na_ratio.get(col,0.0) > 0.80:
            pxd_add(col, global_modes.get(col,'Not Applicable'))

    # Handle Modification slots
    mods = list(dict.fromkeys(pxd_vals.pop('Characteristics[Modification]',[])))
    for i, mod in enumerate(mods):
        slot = 'Characteristics[Modification]' if i==0 else f'Characteristics[Modification].{i}'
        pxd_vals[slot] = [mod]

    # Layer 6: Write per-file
    for i, (row_idx, raw_file) in enumerate(zip(idx, raw_files)):
        fraction = parse_fraction(raw_file)
        biol_rep = parse_biol_replicate(raw_file)
        fn_label = parse_label_from_filename(raw_file)
        fn_conds = parse_condition(raw_file)

        if fraction:
            final_sub.at[row_idx, 'Comment[FractionIdentifier]'] = fraction
        if biol_rep:
            final_sub.at[row_idx, 'Characteristics[BiologicalReplicate]'] = biol_rep
        if fn_label:
            final_sub.at[row_idx, 'Characteristics[Label]'] = fn_label
        for cond_col, cond_val in fn_conds.items():
            if cond_col in final_sub.columns:
                final_sub.at[row_idx, cond_col] = cond_val

        for col in target_cols:
            if final_sub.at[row_idx, col] != 'Not Applicable': continue
            base = re.sub(r'\.\d+$','',col)
            vals = pxd_vals.get(col) or pxd_vals.get(base) or []
            vals = [v for v in vals if str(v).strip().lower() not in ('not applicable','')]
            if vals:
                final_sub.at[row_idx, col] = vals[i % len(vals)]

# Cleanup
final_sub = final_sub.fillna('Not Applicable')
for col in target_cols:
    mask = final_sub[col].astype(str).str.strip().isin(
        ['nan','None','[]','','null','not available','TextSpan'])
    final_sub.loc[mask, col] = 'Not Applicable'

# FractionIdentifier fix
for pxd, grp in final_sub.groupby('PXD'):
    fracs = grp['Comment[FractionIdentifier]'].unique()
    if len(fracs)==1 and str(fracs[0]).strip() in ('1','1.0'):
        final_sub.loc[grp.index,'Comment[FractionIdentifier]'] = 'Not Applicable'

final_sub.to_csv(OUT_PATH, index=False)
print(f'Saved → {OUT_PATH}')
print(f'Shape : {final_sub.shape}')

In [ ]:
label_cols = [c for c in final_sub.columns
              if c not in ('ID','PXD','Raw Data File','Usage')]
rows = [(c,(final_sub[c]!='Not Applicable').sum()) for c in label_cols]
rows.sort(key=lambda x: -x[1])

print(f'{"Column":<55} {"Filled":>7} {"Pct":>6}')
print('-'*72)
for col, n in rows:
    if n > 0:
        print(f'{col:<55} {n:>7} {n/len(final_sub)*100:>5.1f}%')

filled = sum(1 for _,n in rows if n>0)
print(f'\nFilled: {filled} / {len(rows)} columns')

print('\nPubMedBERT contributions (PXDs filled per column):')
for col, n in sorted(bert_stats.items(), key=lambda x: -x[1]):
    print(f'  {col:<50} {n} PXDs')

print('\nSciSpacy NER contributions:')
for col, n in sorted(ner_stats.items(), key=lambda x: -x[1]):
    print(f'  {col:<50} {n} mentions')